# 阶段 2：独立雾密度 Benchmark
此 notebook 自带当前代码，无需远程 Git 分支或访问本地 Windows 路径。不会训练模型。
先运行代码检查，再完成最小闭环、128 样本试验及 CPU/GPU 性能测试。输出是相对响应，不是物理浓度。
仅一个地理区域的四种混合地物 ROI，不能证明跨区域泛化。GPU 缺失明确 NOT RUN。
本文件未预填任何 Colab 结果；运行时间取决于你分配到的硬件。

In [ ]:
# 在 VS Code 右上角选择 Colab 内核，再运行本单元。
import base64, hashlib, io, pathlib, tempfile, zipfile, importlib.util
if not pathlib.Path('/content').is_dir():
    raise RuntimeError('NOT RUN: 当前不是 Colab /content 环境，请选择 Colab 内核。')
missing = [m for m in ['torch', 'numpy', 'PIL', 'yaml', 'matplotlib'] if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError('NOT RUN: 缺少依赖 ' + ', '.join(missing) + '；安装 torch numpy pillow pyyaml matplotlib 后重试。')
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '87e8abda75345ff853cfdb34bf0ed77964b0996d596ccb3842a9b9fd0ab11079'
PROJECT = pathlib.Path(tempfile.mkdtemp(prefix='stage2_haze_', dir='/content'))
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for entry in archive.infolist():
        target = (PROJECT / entry.filename).resolve()
        if not target.is_relative_to(PROJECT):
            raise ValueError('Invalid bundle member')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(entry))
print('项目代码已解包：', PROJECT)


In [ ]:
# 优先探测常见数据目录；找不到时只下载官方小样本，硬上限 24 MiB，不下载整个数据集。
# 若已有数据，可加入 ['--data-root', '/content/drive/MyDrive/你的数据目录']。
# 修复后训练权重必须明确指定 --checkpoint；未指定则学生正确性 NOT EVALUATED。
import subprocess, sys
command = [sys.executable, '-B', '-X', 'utf8', 'scripts/benchmark/run_colab_benchmark.py', '--fetch-small-sample']
result = subprocess.run(command, cwd=PROJECT)
if result.returncode:
    raise RuntimeError('Benchmark 未完整完成，请检查上方日志和 run_status.json。')


In [ ]:
from IPython.display import display, Markdown, Image
for path in sorted((PROJECT / 'experiments').glob('colab_benchmark_*/full_results/report.md')):
    display(Markdown(path.read_text().split('## 固定尺度可视化')[0]))
    for figure in sorted((path.parent / 'figures').glob('comparison_*.png'))[:4]:
        display(Image(filename=str(figure)))
for path in sorted((PROJECT / 'experiments').glob('colab_benchmark_*/performance/report.md')):
    display(Markdown(path.read_text()))
print('保存整个结果目录后再关闭 Colab：', PROJECT / 'experiments')
# 可选：在新的单元运行以下两行打包下载（可能包含较大的合成真值数组）。
# import shutil; bundle = shutil.make_archive('/content/haze_benchmark_results', 'zip', PROJECT / 'experiments')
# from google.colab import files; files.download(bundle)
